In [16]:
#!pip install networkx

In [17]:
import numpy as np
import math
import networkx as nx
import matplotlib.pyplot as plt 
from itertools import combinations
import pandas as pd

In [18]:
historical_data = pd.read_csv("files/all_except_last_orders.csv")
last_orders_data = pd.read_csv("files/last_orders_subset.csv")

In [19]:
historical_data_grouped_by_user = historical_data.groupby('Member')['SKU'].apply(list).reset_index()
historical_data_grouped_by_user.head()

,Member,SKU
0,SSCEHNS,"[15668375, 15668467, 15669863, 15669778, 15669..."
1,SSCESNS,"[15668465, 15668378, 15668453, 15668377, 15669..."
2,SSCEWZO,"[15668474, 15668379, 15668381, 15668688, 15668..."
3,SSCHNCE,"[15668379, 15669767, 15669775, 15669789, 15669..."
4,SSCLCSW,"[15668451, 15668460, 15668685, 15668468, 15669..."


In [20]:
# Build a separate graph for EACH user's purchase history
# IMPROVED: Use order-level co-occurrence instead of treating all purchases as one basket

def build_user_graph_order_based(member):
    """
    Build a co-occurrence graph from individual orders for better temporal patterns.
    
    Args:
        member: User ID to build graph for
        
    Returns:
        NetworkX graph with items as nodes and order-level co-occurrence as edges
    """
    user_graph = nx.Graph()
    
    # Get all orders for this member
    member_data = historical_data[historical_data['Member'] == member]
    
    # Build graph from each order separately
    for order_id in member_data['Order'].unique():
        order_skus = member_data[member_data['Order'] == order_id]['SKU'].tolist()
        
        # Add co-occurrence edges within this specific order
        for item1, item2 in combinations(order_skus, 2):
            if user_graph.has_edge(item1, item2):
                user_graph[item1][item2]['weight'] += 1
            else:
                user_graph.add_edge(item1, item2, weight=1)
    
    return user_graph


# Create a mapping: Member -> User Graph
user_graphs = {}
for member in historical_data['Member'].unique():
    user_graphs[member] = build_user_graph_order_based(member)

print(f"Created {len(user_graphs)} user-specific graphs (order-level co-occurrence)") 

Created 638 user-specific graphs (order-level co-occurrence)


In [ ]:
# Compute PERSONALIZED PageRank for each user's graph
# IMPROVED: Uses personalization vector to teleport back to basket items

def recommend_for_user_ppr(member, user_graph, last_order_skus, top_n=5):
    """
    Generate recommendations using Personalized PageRank (PPR).
    
    PPR gives higher importance to items closely connected to the current basket,
    improving relevance compared to standard PageRank.
    
    Args:
        member: User ID
        user_graph: NetworkX graph for this user
        last_order_skus: List of SKUs in the user's last order
        top_n: Number of recommendations to return
        
    Returns:
        List of recommended SKU IDs
    """
    # Check if graph has enough nodes
    if len(user_graph.nodes()) < 2:
        return []
    
    # Create personalization vector: teleport back to items in current basket
    personalization = {}
    basket_items = [sku for sku in last_order_skus if sku in user_graph.nodes()]
    
    if basket_items:
        # Distribute teleportation probability only among basket items
        for node in user_graph.nodes():
            personalization[node] = 1.0 if node in basket_items else 0.0
        
        # Normalize to sum to 1
        total = sum(personalization.values())
        if total > 0:
            personalization = {k: v/total for k, v in personalization.items()}
        
        try:
            # Personalized PageRank with teleportation to basket items
            pagerank_scores = nx.pagerank(user_graph, personalization=personalization, weight='weight')
        except:
            # Fallback to standard PageRank
            pagerank_scores = nx.pagerank(user_graph, weight='weight')
    else:
        # No basket items in graph, use standard PageRank
        try:
            pagerank_scores = nx.pagerank(user_graph, weight='weight')
        except:
            pagerank_scores = {node: 1.0 / len(user_graph.nodes()) for node in user_graph.nodes()}
    
    # Remove items already in the last order
    filtered_scores = {sku: score for sku, score in pagerank_scores.items() if sku not in last_order_skus}
    
    # Sort by PageRank score (descending) and take top N
    recommended_skus = sorted(filtered_scores.keys(), key=lambda x: filtered_scores[x], reverse=True)[:top_n]
    
    return recommended_skus


# Group last orders by Member
last_orders_grouped = last_orders_data.groupby('Member')['SKU'].apply(list).reset_index()

# Generate recommendations for each user
recommendations = []

for _, row in last_orders_grouped.iterrows():
    member = row['Member']
    last_order_skus = row['SKU']
    
    # Get this user's graph (if it exists)
    if member in user_graphs:
        user_graph = user_graphs[member]
        recommended_skus = recommend_for_user_ppr(member, user_graph, last_order_skus, top_n=5)
        
        recommendations.append({
            'Member': member,
            'Last_Order_SKUs': last_order_skus,
            'Recommended_SKUs': recommended_skus
        })
    else:
        # User not found in historical data (cold start)
        recommendations.append({
            'Member': member,
            'Last_Order_SKUs': last_order_skus,
            'Recommended_SKUs': []
        })

# Convert to DataFrame
recommendations_df = pd.DataFrame(recommendations)
recommendations_df.head()

In [ ]:
# Create a mapping of Order to Member from the original last_orders_data
order_member_map = last_orders_data[['Order', 'Member']].drop_duplicates()

# Merge recommendations with Order IDs
last_orders_with_order = last_orders_data.groupby(['Order', 'Member'])['SKU'].apply(list).reset_index()

# Get global popular items as fallback
global_popular_skus = historical_data['SKU'].value_counts().head(20).index.tolist()

# Create final submission DataFrame
submission_rows = []

for _, row in last_orders_with_order.iterrows():
    order_id = row['Order']
    member = row['Member']
    last_order_skus = row['SKU']
    
    # Get recommendations for this user
    if member in user_graphs:
        user_graph = user_graphs[member]
        recommended_skus = recommend_for_user_ppr(member, user_graph, last_order_skus, top_n=5)
    else:
        recommended_skus = []
    
    # FALLBACK: If we don't have 5 recommendations, add popular items
    if len(recommended_skus) < 5:
        # Add popular items that are NOT in the last order and NOT already recommended
        existing = set(last_order_skus + recommended_skus)
        for popular_sku in global_popular_skus:
            if popular_sku not in existing:
                recommended_skus.append(popular_sku)
                if len(recommended_skus) >= 5:
                    break
    
    # Ensure exactly 5 recommendations (trim if more, though unlikely)
    recommended_skus = recommended_skus[:5]
    
    # Add each recommendation as a separate row
    for sku in recommended_skus:
        submission_rows.append({
            'Order': order_id,
            'SKU': sku,
            'Member': member
        })

# Create submission DataFrame
submission_df = pd.DataFrame(submission_rows)

# Add sequential ID column
submission_df.reset_index(drop=True, inplace=True)
submission_df['ID'] = submission_df.index + 1

# Reorder columns
submission_df = submission_df[['ID', 'Order', 'SKU', 'Member']]

# Save to CSV
submission_df.to_csv('files/submission_pagerank.csv', index=False)

# Display first few rows
print(f"Generated {len(submission_df)} recommendations for {submission_df['Order'].nunique()} orders")
print(f"Expected: {submission_df['Order'].nunique() * 5} recommendations")
submission_df.head(10)

Generated 3190 recommendations for 638 orders
Expected: 3190 recommendations


,ID,Order,SKU,Member
0,1,7341985,15669861,SWCCWNZ
1,2,7341985,34985989,SWCCWNZ
2,3,7341985,15668457,SWCCWNZ
3,4,7341985,15669878,SWCCWNZ
4,5,7341985,15668459,SWCCWNZ
5,6,7344710,15669777,SWRHERW
6,7,7344710,15669818,SWRHERW
7,8,7344710,15669869,SWRHERW
8,9,7344710,7580802,SWRHERW
9,10,7344710,15668452,SWRHERW
